# 1. Stream rows from a parquet file

In [ ]:
from pathlib import Path
from hephaes.parquet import stream_wide_parquet_rows

parquet_path = Path("output") / "episode_0001.parquet"

for idx, row in enumerate(stream_wide_parquet_rows(parquet_path), start=1):
    print(f"Row {idx}: {row}")
    if idx >= 5:
        break


# 2. Extract images from a parquet file

In [ ]:
from pathlib import Path
from hephaes.parquet import extract_images

parquet_path = Path("output") / "episode_0001.parquet"
image_column = "left_camera_image_compressed"
save_dir = Path("output") / "extracted_images"
save_dir.mkdir(parents=True, exist_ok=True)

images = extract_images(parquet_path, image_column)

def infer_extension(data: bytes) -> str:
    if data.startswith(b"\xff\xd8\xff"):
        return ".jpg"
    if data.startswith(b"\x89PNG\r\n\x1a\n"):
        return ".png"
    return ".bin"

for idx, image_bytes in enumerate(images, start=1):
    ext = infer_extension(image_bytes)
    output_path = save_dir / f"{image_column}_{idx:04d}{ext}"
    output_path.write_bytes(image_bytes)

print(f"Saved {len(images)} images to {save_dir.resolve()}")


In [ ]:
from pathlib import Path
from rosbags.typesys import Stores, get_typestore

image_dir = Path("output") / "extracted_images"
typestore = get_typestore(Stores.ROS2_HUMBLE)

written = 0
for bin_path in sorted(image_dir.glob("*.bin")):
    msg = typestore.deserialize_cdr(
        bin_path.read_bytes(),
        "sensor_msgs/msg/CompressedImage",
    )
    fmt = msg.format.lower()
    ext = ".jpg" if "jpeg" in fmt or "jpg" in fmt else ".png"
    out_path = bin_path.with_suffix(ext)
    out_path.write_bytes(bytes(msg.data))
    written += 1

print(f"Wrote {written} decoded images to {image_dir.resolve()}")
